# Test preprocessing pipeline

In [1]:
import sys
import pandas as pd
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.dataset.data_pipeline import clean_data, split_data, verify_stratification

In [2]:
RAW_PATH = Path("../data/raw/adult_raw.csv")

adult_df = pd.read_csv(RAW_PATH)

adult_df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [3]:
print(f"Rows:    {adult_df.shape[0]:,}")
print(f"Columns: {adult_df.shape[1]}")
print()


summary = pd.DataFrame({
    "dtype":          adult_df.dtypes,
    "unique_values":  adult_df.nunique(),
    "sample_value":   adult_df.iloc[0],
})

summary

Rows:    48,842
Columns: 15



,dtype,unique_values,sample_value
age,int64,74,39
workclass,str,9,State-gov
fnlwgt,int64,28523,77516
education,str,16,Bachelors
education-num,int64,16,13
marital-status,str,7,Never-married
occupation,str,15,Adm-clerical
relationship,str,6,Not-in-family
race,str,5,White
sex,str,2,Male


In [4]:
adult_df.isnull().sum()

age                 0
workclass         963
fnlwgt              0
education           0
education-num       0
marital-status      0
occupation        966
relationship        0
race                0
sex                 0
capital-gain        0
capital-loss        0
hours-per-week      0
native-country    274
income              0
dtype: int64

## Clean data

In [5]:
adult_cleaned_df = clean_data(adult_df)

File saved successfully to: F:\Documents\HSLU\Studium\Workspaces\BAA\code\data\cleaned\adult_cleaned.csv


In [6]:
print(f"Rows:    {adult_cleaned_df.shape[0]:,}")
print(f"Columns: {adult_cleaned_df.shape[1]}")
print()


cleaned_summary = pd.DataFrame({
    "dtype":          adult_cleaned_df.dtypes,
    "unique_values":  adult_cleaned_df.nunique(),
    "sample_value":   adult_cleaned_df.iloc[0],
})

cleaned_summary

Rows:    48,842
Columns: 13



,dtype,unique_values,sample_value
age,int64,74,39
workclass,str,8,State-gov
education-num,int64,16,13
marital-status,str,7,Never-married
occupation,str,14,Adm-clerical
relationship,str,6,Not-in-family
race,str,5,White
sex,str,2,Male
capital-gain,int64,123,2174
capital-loss,int64,99,0


In [7]:
target_counts = adult_cleaned_df["income"].value_counts()
target_pct = adult_cleaned_df["income"].value_counts(normalize=True).mul(100).round(2)

target_dist = pd.DataFrame({
    "count": target_counts,
    "percent": target_pct
})

print(target_dist)

        count  percent
income                
<=50K   37155    76.07
>50K    11687    23.93


In [8]:
adult_cleaned_df.isnull().sum().sort_values(ascending=False)

occupation        2809
workclass         2799
native-country     857
education-num        0
age                  0
marital-status       0
relationship         0
sex                  0
race                 0
capital-gain         0
capital-loss         0
hours-per-week       0
income               0
dtype: int64

## Split data

In [9]:
train_df, val_df, test_df = split_data(adult_cleaned_df, "income", random_state=42)

In [10]:
verify_stratification(train_df, val_df, test_df, "income")

,Train,Val,Test
income,,,
<=50K,76.07,76.08,76.07
>50K,23.93,23.92,23.93
